# 05 — Entraînement dans les conditions de production

**Projet** : Prédiction d'attrition client (churn télécom) avec LightGBM
**Ce que fait `make train`** : charger → valider → splitter → features → preprocessing →
entraîner → métriques de validation → **persister** (modèle, fiche, métriques, splits).

Ce notebook exécute exactement le même chemin, mais de façon instrumentée : chaque brique est
visible, mesurable et rejouable.

## Objectifs pédagogiques

1. Piloter un entraînement par **objets** (Trainer, callbacks) plutôt que par un script monolithique.
1. Lire un `TrainingOutcome` : métriques, durée, historique, artefacts.
1. Mesurer la **stabilité** d'un modèle (plusieurs graines) avant de conclure.
1. Vérifier le garde-fou de qualité déclaré en configuration.

**Objectifs transverses du dépôt**

- Comprendre la croissance leaf-wise (`num_leaves`) et son risque de sur-apprentissage face à `max_depth`.
- Armer l'early stopping natif via un callback LightGBM (`early_stopping`, `log_evaluation`) sans réécrire la boucle.
- Régulariser un booster (min_child_samples, subsample + subsample_freq, colsample_bytree, reg_lambda, min_split_gain).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.7)")
print(f"Cible             : {CONFIG.data.target}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. Le modèle, construit depuis la configuration

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)
print(MODEL.summary())
pd.Series(FIT_RESULT.metrics, name="métrique").to_frame("valeur")

**Ce qu'il faut retenir**

- `build_model` est la **seule** porte d'entrée : le reste du projet ne connaît que `BaseModel`.
- Changer de stack (XGBoost, PyTorch, …) ne modifie ni le trainer, ni l'évaluateur, ni l'inférence.
- Les hyperparamètres viennent de `conf/model/default.yaml` : aucun n'est codé en dur ici.

## 2. Callbacks : instrumenter sans polluer la boucle d'entraînement

In [ ]:
from src.training.callbacks import (
    EarlyStoppingCallback,
    LoggingCallback,
    MetricHistoryCallback,
    MetricThresholdCallback,
)
from src.training.trainer import Trainer, TrainingData

TRAINING_DATA = TrainingData(
    X_train=PREPARED["X_train"],
    y_train=PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    feature_names=PREPARED["feature_names"],
    task=CONFIG.metrics.task,
)

CALLBACKS = [
    LoggingCallback(every=1),
    MetricHistoryCallback(),
    EarlyStoppingCallback(
        monitor=CONFIG.train.early_stopping.monitor,
        patience=CONFIG.train.early_stopping.patience,
        mode=CONFIG.train.early_stopping.mode,
    ),
    MetricThresholdCallback(
        monitor=f"val_{CONFIG.metrics.primary}",
        threshold=float(CONFIG.metrics.min_primary or 0.0),
        mode="max" if CONFIG.metrics.direction == "maximize" else "min",
    ),
]
[callback.name for callback in CALLBACKS]

In [ ]:
TRAINER = Trainer(
    MODEL,
    config=CONFIG.model_dump(),
    paths=NB_PATHS,
    metric_names=CONFIG.metrics.all_metrics,
    task=CONFIG.metrics.task,
    callbacks=CALLBACKS,
)
OUTCOME = TRAINER.train(TRAINING_DATA)

metrics_frame = pd.DataFrame(
    {
        "métrique": list(OUTCOME.metrics),
        "valeur": [OUTCOME.metrics[name] for name in OUTCOME.metrics],
    }
).sort_values("valeur", ascending=False)
print(f"durée : {OUTCOME.duration_seconds:.2f}s | artefacts : {len(OUTCOME.artifacts)}")
metrics_frame.round(4).reset_index(drop=True)

**Ce qu'il faut retenir**

- Les métriques préfixées `val_` viennent du **split de validation** : elles ne sont jamais calculées sur le test.
- Le test reste vierge jusqu'au notebook 06 — c'est la condition d'une estimation honnête.
- La durée est tracée : un entraînement qui double soudainement signale une dérive de données ou de configuration.

## 3. Artefacts produits

In [ ]:
import json

artifacts = pd.DataFrame(
    {
        "artefact": list(OUTCOME.artifacts),
        "chemin": [OUTCOME.artifacts[name] for name in OUTCOME.artifacts],
    }
)
display(artifacts)

card_path = OUTCOME.artifacts.get("model_card")
if card_path:
    card = json.loads(Path(card_path).read_text(encoding="utf-8"))
    print("fiche modèle — clés :", sorted(card))
    print("features attendues  :", len(card["feature_names"]))
    print("versions librairies :", card["library_versions"])

**Ce qu'il faut retenir**

- La **fiche modèle** (features, paramètres, versions, métriques) rend un artefact reproductible et auditable.
- Sans elle, un `.joblib` retrouvé dans six mois est inutilisable : impossible de savoir quoi lui donner en entrée.
- Les artefacts du notebook sont écrits dans `outputs/notebooks` ; `make train` écrit dans `artifacts/`.

## 4. Stabilité : plusieurs graines, même conclusion ?

In [ ]:
from src.training.losses_metrics import MetricCalculator, MetricInputs

scores = []
for seed in (7, 21, 42):
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"])
    candidate.random_state = seed
    _ = candidate.fit(
        PREPARED["X_train"],
        PREPARED["y_train"],
        X_val=PREPARED["X_val"],
        y_val=PREPARED["y_val"],
        callbacks=[],
    )
    values = MetricCalculator(task=CONFIG.metrics.task, metrics=[CONFIG.metrics.primary]).evaluate(
        MetricInputs(
            y_true=PREPARED["y_val"],
            y_pred=candidate.predict(PREPARED["X_val"]),
            y_proba=candidate.predict_proba(PREPARED["X_val"])
            if candidate.supports_proba
            else None,
        )
    )
    scores.append(values.get(CONFIG.metrics.primary, float("nan")))

scores_array = np.asarray(scores, dtype="float64")
stability = pd.DataFrame(
    {
        "graine": [7, 21, 42],
        CONFIG.metrics.primary: scores_array.round(4),
    }
)
mean = float(np.nanmean(scores_array))
std = float(np.nanstd(scores_array, ddof=1)) if len(scores_array) > 1 else 0.0
print(f"moyenne = {mean:.4f} | ecart-type = {std:.4f}")
print(f"intervalle +/- 1 ecart-type = [{mean - std:.4f} ; {mean + std:.4f}]")
stability

**Ce qu'il faut retenir**

- Un écart-type élevé signifie que le résultat dépend de la graine : toute comparaison de modèles doit le mesurer.
- Règle pratique : ne pas célébrer un gain inférieur à 2 x l'écart-type observé.
- Cette variabilité est aussi un argument pour la **validation croisée** en CI (`train.cross_validation`).

## 5. Garde-fou de qualité

In [ ]:
threshold_callback = next(
    (callback for callback in CALLBACKS if isinstance(callback, MetricThresholdCallback)), None
)
threshold = CONFIG.metrics.min_primary
observed = OUTCOME.metrics.get(f"val_{CONFIG.metrics.primary}", float("nan"))
verdict = (
    "INCONNU" if not np.isfinite(observed) else ("OK" if observed >= float(threshold) else "ÉCHEC")
)
print(f"métrique          : val_{CONFIG.metrics.primary}")
print(f"valeur observée   : {observed:.4f}")
print(f"seuil configuré   : {threshold}")
print(f"verdict           : {verdict}")
print(f"callback satisfait: {getattr(threshold_callback, 'satisfied', 'n/a')}")

**Ce qu'il faut retenir**

- Le seuil vit dans `conf/config.yaml` (`metrics.min_primary`) : la CI l'utilise comme gate de déploiement.
- Un modèle sous le seuil ne doit **pas** être promu — même s'il « marche » en apparence.

## 6. Rechargement et vérification

In [ ]:
from src.models import load_model

RESTORED = load_model(OUTCOME.artifacts["model"])
original = np.asarray(MODEL.predict(PREPARED["X_test"]), dtype="float64")
reloaded = np.asarray(RESTORED.predict(PREPARED["X_test"]), dtype="float64")
print("modèle rechargé :", RESTORED.summary())
print("prédictions identiques :", bool(np.allclose(original, reloaded)))

**Ce qu'il faut retenir**

- Le test de rechargement est **le** test de déploiement : un modèle qui ne se recharge pas ne se sert pas.
- Il est rejoué automatiquement dans `tests/test_models.py::TestPersistence`.

## Synthèse

| Étape | Objet utilisé | Artefact |
| --- | --- | --- |
| Construction | `build_model(CONFIG)` | — |
| Entraînement | `Trainer.train(TrainingData)` | `model.joblib` |
| Instrumentation | `LoggingCallback`, `MetricHistoryCallback`, `EarlyStoppingCallback`, `MetricThresholdCallback` | historique |
| Traçabilité | `ModelCard` | `model_card.json` |
| Métriques | `MetricCalculator` | `training_metrics.json` |
| Qualité | `metrics.min_primary` | verdict CI |

**Suite** : `06_error_analysis.ipynb` évalue sur le **test** et transforme les erreurs en décisions.